# Feature Analysis for Time to Approval

This notebook computes the statistical significance of selected features in predicting the time to approval for data centers using data from final/data_centers_wide.csv.

In [27]:
import pandas as pd
from scipy import stats
from datetime import datetime

In [28]:
# Load the data
df = pd.read_csv('./final/data_centers_wide.csv')

# Parse date columns (adjust column names if necessary)
date_cols = ['earliest_llm_event_date', 'latest_llm_event_date', 'earliest_air_program_begin', 'latest_air_program_begin', 'earliest_water_orig_issue', 'earliest_water_issue', 'latest_water_issue', 'latest_water_expiration', 'latest_water_termination', 'earliest_known_permit_date']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Compute time_to_approval in days (assuming it's the duration between earliest and latest LLM event)
df['time_to_approval'] = (df['latest_llm_event_date'] - df['earliest_llm_event_date']).dt.days

# Drop rows where time_to_approval is NaN
df = df.dropna(subset=['time_to_approval'])

## Continuous Features Analysis

In [29]:
# Expanded continuous features
features = [
    'mw_capacity', 'facility_size_sqft', 'property_size_acres', 'project_cost',
    'n_llm_events', 'number_of_generators', 'number_of_buildings',
    'echo_distance_miles', 'match_score', 'match_name_score', 'match_addr_score',
    'n_air_programs', 'n_water_permits'
]
target = 'time_to_approval'

In [30]:
# Compute statistical significance using Pearson correlation
for feature in features:
    if feature not in df.columns:
        print(f'Feature {feature} not in data')
        continue
    valid = df[[feature, target]].dropna()
    if len(valid) < 2:
        print(f'Not enough data for {feature}')
        continue
    # Convert to float, coercing errors to NaN and skip invalid rows
    x = pd.to_numeric(valid[feature], errors='coerce')
    y = pd.to_numeric(valid[target], errors='coerce')
    valid_converted = pd.DataFrame({'x': x, 'y': y}).dropna()
    if len(valid_converted) < 2:
        print(f'Not enough valid numeric data for {feature}')
        continue
    corr, p_value = stats.pearsonr(valid_converted['x'], valid_converted['y'])
    print(f'Feature: {feature}')
    print(f'Correlation: {corr:.4f}')
    print(f'P-value: {p_value:.4f}')
    significance = 'Significant' if p_value < 0.05 else 'Not significant'
    print(f'{significance}\n')

Feature: mw_capacity
Correlation: -0.3194
P-value: 0.0154
Significant

Feature: facility_size_sqft
Correlation: 1.0000
P-value: 1.0000
Not significant

Feature: property_size_acres
Correlation: -0.1284
P-value: 0.2101
Not significant

Not enough valid numeric data for project_cost
Feature: n_llm_events
Correlation: -0.1164
P-value: 0.1439
Not significant

Not enough valid numeric data for number_of_generators
Feature: number_of_buildings
Correlation: -0.0738
P-value: 0.7091
Not significant

Feature: echo_distance_miles
Correlation: -0.1286
P-value: 0.1084
Not significant

Feature: match_score
Correlation: 0.1670
P-value: 0.0365
Significant

Feature: match_name_score
Correlation: -0.1271
P-value: 0.1126
Not significant

Feature: match_addr_score
Correlation: 0.2192
P-value: 0.0058
Significant

Feature: n_air_programs
Correlation: 0.0668
P-value: 0.4029
Not significant

Feature: n_water_permits
Correlation: -0.1197
P-value: 0.1330
Not significant



## Categorical Features Analysis

In [31]:
# Expanded categorical features
cat_features = [
    'state', 'fractracker_status', 'purpose', 'power_source', 'dedicated_power_plant',
    'cooling_type', 'community_pushback', 'resistance_status', 'AIR_FLAG', 'NPDES_FLAG',
    'RCRA_FLAG', 'SDWIS_FLAG', 'GHG_FLAG', 'FAC_ACTIVE_FLAG', 'FAC_MAJOR_FLAG',
    'match_naics_hit', 'match_sic_hit', 'air_program_codes', 'water_permit_types'
]

In [32]:
# Analyze categorical features
for feature in cat_features:
    if feature not in df.columns:
        print(f'Feature {feature} not in data')
        continue
    print(f'Feature: {feature}')
    groups = df.groupby(feature)[target].apply(list)
    group_means = df.groupby(feature)[target].mean()
    print('Mean time_to_approval by category:')
    for cat, mean in group_means.items():
        print(f'  {cat}: {mean:.2f} days')
    
    # Filter groups with at least 2 samples
    valid_groups = [g for g in groups if len(g) >= 2]
    if len(valid_groups) < 2:
        print('Not enough groups with sufficient data for statistical test.\n')
        continue
    
    # Perform ANOVA if more than 2 groups, else t-test
    if len(valid_groups) == 2:
        t_stat, p_value = stats.ttest_ind(*valid_groups)
        test_name = 't-test'
    else:
        f_stat, p_value = stats.f_oneway(*valid_groups)
        test_name = 'ANOVA'
    
    print(f'{test_name} p-value: {p_value:.4f}')
    significance = 'Significant' if p_value < 0.05 else 'Not significant'
    print(f'{significance}\n')

Feature: state
Mean time_to_approval by category:
  AL: 405.00 days
  AR: 4780.00 days
  AZ: 778.60 days
  CA: 1498.33 days
  FL: 3582.50 days
  GA: 2001.00 days
  IA: 521.00 days
  IL: 708.50 days
  IN: 542.29 days
  KS: 192.00 days
  KY: 256.00 days
  MA: 1824.00 days
  MD: 3118.25 days
  MI: 513.60 days
  MN: 397.50 days
  MO: 579.50 days
  NC: 145.75 days
  ND: 152.00 days
  NE: 5501.00 days
  NH: 7373.00 days
  NJ: 3741.50 days
  NM: 247.00 days
  NV: 5374.00 days
  NY: 1814.17 days
  OH: 306.00 days
  OK: 634.67 days
  OR: 591.00 days
  PA: 591.13 days
  SC: 421.00 days
  SD: 133.00 days
  TN: 806.33 days
  TX: 1220.75 days
  UT: 1197.00 days
  VA: 925.94 days
  WI: 418.00 days
  WV: 308.00 days
ANOVA p-value: 0.0096
Significant

Feature: fractracker_status
Mean time_to_approval by category:
  Approved/Permitted/Under construction: 895.47 days
  Cancelled: 288.64 days
  Expanding: 1138.20 days
  Operating: 3013.81 days
  Proposed: 912.48 days
  Suspended: 478.25 days
  Unknown: 3